# Feature Engineering & Descriptors for Materials
### FDP: Machine Learning for Materials and Metallurgical Engineering

**Session objectives:**
1. Understand why raw composition isn't usable directly by an ML model
2. Understand the general principle of feature engineering – representing a real-world object as a numeric vector
3. Learn what Magpie-style descriptors are, and how they're computed
4. Build hands-on fluency with `pymatgen` and `matminer`
5. Check that not every generated feature carries predictive power – a materials-specific version of the Day 1 lesson
6. Produce a properly featurized dataset, ready for the classification sessions that follow

**Note on scope:** this session does not teach any classification algorithm in depth – logistic regression and Random Forest each get their own dedicated session later. Any model used here is purely a diagnostic tool.

## The Case Study: Predicting Alloy Phase From Composition

**Given a composition, will it form an FCC, BCC, or mixed/other structure?**

This is a real, actively researched materials-informatics problem, especially for multi-principal-element alloys (MPEAs) and high-entropy alloys (HEAs) – alloys with several elements in high concentration, where traditional metallurgical rules of thumb often break down.

**Our dataset:** Borg, Frey, Moh, Pollock, Gorsse, Miracle, Senkov, Meredig, Saal – *"Expanded dataset of mechanical properties and observed phases of multi-principal element alloys,"* Scientific Data, 7:430, 2020 (Citrine Informatics). **1,545 real alloys**, each with a measured composition and observed phase, compiled from published literature.

## Setup — Install and Import Libraries
`matminer` and `pymatgen` aren't pre-installed on Colab – run this cell first (takes ~30 seconds).

In [ ]:
!pip install matminer pymatgen --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty


## Step 1 — Why Raw Composition Isn't Usable Directly

A composition like `"Al0.5 Co1 Fe1 Ni1"` is just text – no ML model can take a formula string as a numeric input. Before any regression or classification can happen, we need to turn this into numbers.

This is exactly the same problem we solved differently on Day 1: there, we had numeric measurements (grain size) but needed the *right* transform (d raised to the power -1/2). Here, we don't even have numbers yet – we have to build them from scratch out of a chemical formula. **This is feature engineering in its most fundamental form.**

## Step 2 — The General Feature Engineering Principle

Feature engineering means representing a real-world object as a fixed-length numeric vector that a model can actually use.

For a composition, the challenge is that alloys have **different numbers of elements** – a binary alloy has 2, a high-entropy alloy might have 5 or more. A model needs every example to have the *same* number of input features, regardless of how many elements are actually present.

**The standard solution (Magpie-style descriptors):**
1. Look up tabulated physical properties for each individual element (atomic number, melting point, electronegativity, valence electrons, etc.)
2. Combine these per-element values across the whole composition using aggregation statistics: **mean, minimum, maximum, range, mode, average deviation**

The result: every alloy, no matter how many elements it contains, becomes the same fixed-length vector.

## Step 3 — Load the Real Dataset

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vijindal/fdp-ml/main/day2/MPEA_dataset.csv"
df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
df[['FORMULA', 'PROPERTY: BCC/FCC/other']].head()


In [ ]:
print("Phase label distribution:")
print(df['PROPERTY: BCC/FCC/other'].value_counts())


Notice the class imbalance – `other` (mixed/complex phases) is the most common outcome, `FCC` the least. Worth keeping in mind for whichever classifier is trained on this later.

---
## Quick Check 1

Why can't we just assign each element a single ID number (Al=1, Co=2, Fe=3...) and feed that directly into a model?

**(i)  That would work perfectly well**
**(ii)  Arbitrary ID numbers imply a false numeric relationship (e.g., that element “3” is meaningfully “between” elements “2” and “4”) with no physical basis, and still wouldn't handle alloys with different numbers of elements**
**(iii)  Models can't use numbers at all, only text**

*Think about it, then check the answer below.*

**Answer: (ii)** — arbitrary ID encoding creates false ordinal relationships a model might try to learn from, and still doesn't solve the fixed-length problem (a 2-element and a 5-element alloy would need different numbers of ID slots). Magpie-style descriptors solve both problems: they use physically meaningful properties, aggregated into a fixed-length vector regardless of element count.

## Step 4 — Magpie Descriptors, By Hand

Before automating this, let's compute one Magpie feature manually for a real alloy: **Al0.5 Co1 Fe1 Ni1**.

First, convert amounts to fractions of the total:

In [ ]:
from pymatgen.core import Composition

comp = Composition("Al0.5 Co1 Fe1 Ni1")
fractions = comp.fractional_composition.as_dict()
print("Composition fractions:", fractions)


Now, using tabulated atomic numbers (Al=13, Co=27, Fe=26, Ni=28), compute the **composition-weighted mean atomic number** by hand:

$$\text{mean} = 0.143(13) + 0.286(27) + 0.286(26) + 0.286(28) = 25.0$$

In [ ]:
atomic_numbers = {'Al': 13, 'Co': 27, 'Fe': 26, 'Ni': 28}
manual_mean = sum(fractions[el] * atomic_numbers[el] for el in fractions)
print(f"Manually computed mean atomic number: {manual_mean:.2f}")


Now let's confirm `matminer` computes exactly the same thing automatically:

In [ ]:
featurizer = ElementProperty.from_preset("magpie")
result = featurizer.featurize(comp)
labels = featurizer.feature_labels()
feature_dict = dict(zip(labels, result))

print("matminer's 'MagpieData mean Number':", feature_dict['MagpieData mean Number'])
print("Matches our manual calculation:", np.isclose(manual_mean, feature_dict['MagpieData mean Number']))


---
## Quick Check 2

`matminer` also computes “MagpieData avg_dev Number” (average deviation from the mean) for this same alloy. What does a *large* average deviation tell you about a composition?

**(i)  The alloy contains elements with very similar atomic properties**
**(ii)  The alloy contains elements whose properties differ substantially from each other**
**(iii)  The alloy has a manufacturing defect**

*Think about it, then check the answer below.*

**Answer: (ii)** — average deviation measures spread. A high value means the constituent elements are quite different from each other in that property; a low value means they're similar. This single number can capture something about how “mismatched” an alloy's elements are – often physically relevant to phase stability.

## Step 5 — Automating This for All 1,545 Alloys

Doing this by hand for every alloy and every property would be impossibly slow. This is exactly what `matminer` automates.

In [ ]:
df = df.dropna(subset=['FORMULA', 'PROPERTY: BCC/FCC/other']).copy()
df['composition'] = df['FORMULA'].apply(Composition)

featurizer = ElementProperty.from_preset("magpie")
df_featurized = featurizer.featurize_dataframe(df, col_id='composition', ignore_errors=True)

feature_cols = featurizer.feature_labels()
print(f"Generated {len(feature_cols)} Magpie features for {len(df_featurized)} alloys")
df_featurized[feature_cols[:6]].head()


## Step 6 — Not Every Feature Has Predictive Power

132 features is a lot. Just like Day 1's lesson on feature predictive power, not all of them will actually help predict phase. Let's do a **lightweight diagnostic check** – not a real model-training exercise (that's for the classification sessions), just a quick look at which features carry the most signal.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df_featurized[feature_cols].fillna(df_featurized[feature_cols].mean())
y = df_featurized['PROPERTY: BCC/FCC/other']

# Diagnostic only -- trained on all data, no train/test split.
# Proper evaluation methodology is covered in the classification sessions.
diagnostic_rf = RandomForestClassifier(n_estimators=200, random_state=42)
diagnostic_rf.fit(X, y)

importances = pd.Series(diagnostic_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top 10 most predictive features (diagnostic only):")
print(importances.head(10))


In [ ]:
plt.figure(figsize=(7, 5))
importances.head(10).sort_values().plot(kind='barh', color='navy')
plt.xlabel("Feature Importance (diagnostic)")
plt.title("Which Magpie Features Carry the Most Signal?")
plt.tight_layout()
plt.show()


**Worth noticing:** several top features relate to valence electrons (`NdValence`, `NValence`, `NUnfilled`). This isn't a coincidence – valence electron concentration (VEC) is a well-known empirical rule real metallurgists already use to predict HEA phase formation (roughly: low VEC favors BCC, high VEC favors FCC). Our purely data-driven feature ranking independently rediscovered a physically meaningful signal that domain scientists already knew about – a reassuring sign the features are capturing something real, not noise.

---
## Quick Check 3

Why is it reassuring that the top data-driven features relate to valence electron count, a quantity metallurgists already use empirically?

**(i)  It proves Random Forest is the best possible classifier for this problem**
**(ii)  It suggests the engineered features are capturing physically meaningful signal, not just fitting noise – independent confirmation from domain knowledge**
**(iii)  It means Er and H should have been used instead**

*Think about it, then check the answer below.*

**Answer: (ii)** — when a purely statistical feature-importance ranking lines up with an independently-known physical rule, that's a strong, reassuring signal the features (and the data) are sound – the same kind of sanity check we used in Session 1 of the clustering topic ('does this match domain knowledge?').

## Step 7 — Save the Featurized Dataset

This properly featurized dataset – 132 numeric descriptors per alloy, plus the true phase label – is now ready for the classification sessions that follow.

In [ ]:
output_cols = ['FORMULA', 'PROPERTY: BCC/FCC/other'] + feature_cols
df_featurized[output_cols].to_csv('MPEA_featurized.csv', index=False)
print("Saved MPEA_featurized.csv:", df_featurized[output_cols].shape)


## Wrap-Up

- Raw composition (a formula string) can't be used directly by any ML model – it has to become a fixed-length numeric vector first
- Magpie descriptors solve this generally: tabulated per-element properties, aggregated via mean/min/max/range/mode/avg_dev, regardless of how many elements are in the alloy
- We verified `matminer`'s automation against a hand-computed example – exact match
- Not every one of the 132 generated features is equally useful – a quick diagnostic check showed valence-electron-related features ranking highest, independently matching known metallurgical intuition
- The featurized dataset is now saved and ready

**Next: logistic regression and Random Forest, each in their own dedicated session, using this exact featurized dataset.**